
## CELEBAL TECHNOLOGIES

#### Data Engineering Internship (CEI Program 2026) Week 7 Assignment

#### File - Delta Operation
---

#### Submitted By
**Harshit Sharma**  
*Data Engineering Intern*

---


In [0]:
from pyspark.sql.functions import *

### **1**. Upload CSV to Volume

In [0]:
raw_path = "/Volumes/dlt_assignement/default/assignment_vol/RawVol/Superstore.csv"

df_superstore_bronze = spark.read.csv(
    raw_path,
    header=True,
    inferSchema=True
)

df_superstore_bronze.limit(5).display()
print("Row count:", df_superstore_bronze.count())

Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.96,2,0,41.9136
2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs, Rounded Back",731.94,3,0,219.582
3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters by Universal,14.62,2,0,6.8714
4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.031
5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.368,2,0.2,2.5164


Row count: 9994


In [0]:
df_superstore_bronze.printSchema()

root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: date (nullable = true)
 |-- Ship Date: date (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- Discount: string (nullable = true)
 |-- Profit: double (nullable = true)



 Writing Delta Table In **dlt_assignment Catalog**

In [0]:
df_superstore_bronze.write.format("delta") \
    .mode("overwrite") \
    .option("delta.columnMapping.mode", "name") \
    .saveAsTable("dlt_assignement.default.superstore_bronze")

In [0]:
%sql
Select Count(*) from dlt_assignement.default.superstore_bronze

Count(*)
9994


In [0]:
# Read from Bronze table
df_bronze = spark.table("dlt_assignement.default.superstore_bronze")

In [0]:

# Check nulls per column before cleaning
from pyspark.sql.functions import col, sum as spark_sum

null_counts = df_bronze.select(
    [spark_sum(col(c).isNull().cast("int")).alias(c) for c in df_bronze.columns]
)
display(null_counts)

Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [0]:
# Check duplicate count before cleaning
total_rows = df_bronze.count()
distinct_rows = df_bronze.distinct().count()
print("Total rows:", total_rows)
print("Distinct rows:", distinct_rows)
print("Duplicate rows:", total_rows - distinct_rows)

Total rows: 9994
Distinct rows: 9994
Duplicate rows: 0


**Drop** exact duplicate rows

In [0]:

df_silver = df_bronze.dropDuplicates()

In [0]:
# drop rows where Null values present In (Order ID, Product ID, Customer ID)
df_silver = df_silver.dropna(subset=["Order ID", "Product ID", "Customer ID"])

In [0]:
print("Rows after cleaning:", df_silver.count())

Rows after cleaning: 9994


In [0]:
df_silver.write.format("delta")\
    .mode("overwrite")\
    .option("delta.columnMapping.mode", "name")\
    .saveAsTable("dlt_assignement.default.superstore_silver")

### Load incremental dataset

In [0]:
incremental_path = "/Volumes/dlt_assignement/default/assignment_vol/RawVol/Incremental_load.csv"

df_incremental = spark.read.csv(
    incremental_path,
    header=True,
    inferSchema=True
)
print("Row count:", df_incremental.count())
df_incremental.printSchema()

Row count: 9998
root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- Discount: string (nullable = true)
 |-- Profit: double (nullable = true)



In [0]:
silver_cols = set(spark.table("dlt_assignement.default.superstore_silver").columns)
incr_cols = set(df_incremental.columns)

print("Columns only in Silver:", silver_cols - incr_cols)
print("Columns only in Incremental:", incr_cols - silver_cols)

Columns only in Silver: set()
Columns only in Incremental: set()


In [0]:
from pyspark.sql.functions import col, coalesce, expr

df_incremental_fixed = df_incremental \
    .withColumn("Order Date", coalesce(
        expr("try_to_date(`Order Date`, 'MM-dd-yyyy')"),
        expr("try_to_date(`Order Date`, 'M/d/yyyy')")
    )) \
    .withColumn("Ship Date", coalesce(
        expr("try_to_date(`Ship Date`, 'MM-dd-yyyy')"),
        expr("try_to_date(`Ship Date`, 'M/d/yyyy')")
    ))

In [0]:
from delta.tables import DeltaTable

silver_table = DeltaTable.forName(spark, "dlt_assignement.default.superstore_silver")

silver_table.alias("target").merge(
    df_incremental_fixed.alias("source"),
    "target.`Row ID` = source.`Row ID`"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

print("Merge completed.")

Merge completed.


In [0]:
# Get merge operation metrics
history_df = spark.sql("DESCRIBE HISTORY dlt_assignement.default.superstore_silver LIMIT 1")
display(history_df.select("operation", "operationMetrics"))

operation,operationMetrics
MERGE,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 3, numTargetBytesAdded -> 420646, numTargetBytesRemoved -> 339834, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 9994, executionTimeMs -> 5610, materializeSourceTimeMs -> 454, numTargetRowsInserted -> 4, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 2891, numTargetRowsUpdated -> 9994, numOutputRows -> 9998, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 9998, numTargetFilesRemoved -> 1, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 2135)"


In [0]:
metrics = history_df.select("operationMetrics").collect()[0]["operationMetrics"]

updated = metrics.get("numTargetRowsUpdated")
inserted = metrics.get("numTargetRowsInserted")

total_after = spark.table("dlt_assignement.default.superstore_silver").count()

print(f"Total records after merge : {total_after}")
print(f"Records updated           : {updated}")
print(f"Records inserted          : {inserted}")

Total records after merge : 9998
Records updated           : 9994
Records inserted          : 4


In [0]:
%sql
select Count(*) as Number_of_records
from dlt_assignement.default.superstore_silver

Number_of_records
9998
